In [ ]:
import pandas as pd
import numpy as np

In [ ]:
cols_needed = ["order",	"species", "speciesKey", "isInvasive", "decimalLatitude", "decimalLongitude",
               "eventDate", "day", "eventTime", "elevation", "basisOfRecord", "coordinateUncertaintyInMeters", "hasCoordinate", "hasGeospatialIssues", "iucnRedListCategory"]
dtypes = {"decimalLatitude": "float32", "decimalLongitude": "float32"}

chunks = []
for chunk in pd.read_csv('/content/drive/MyDrive/occurrence.txt', sep='\t',
                          usecols=cols_needed, dtype=dtypes,
                          chunksize=200_000, low_memory=False):
    # chunk = chunk[chunk["basisOfRecord"].isin(["HUMAN_OBSERVATION", "MACHINE_OBSERVATION"])]
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
print(df.shape)

(830240, 15)


In [ ]:
df.head()

,basisOfRecord,eventDate,eventTime,day,decimalLatitude,decimalLongitude,coordinateUncertaintyInMeters,order,elevation,hasCoordinate,hasGeospatialIssues,speciesKey,species,isInvasive,iucnRedListCategory
0,HUMAN_OBSERVATION,2024-01-08T09:41:10,09:41:10-05:00,8.0,46.232197,-67.793869,34.0,Hemiptera,NaN,True,false,6PGH7,Leptoglossus phyllopus,NaN,NaN
1,HUMAN_OBSERVATION,2024-01-24T17:54,17:54:00-05:00,24.0,43.022518,-77.060684,15.0,Lepidoptera,NaN,True,false,943K8,Hypena scabra,NaN,NaN
2,HUMAN_OBSERVATION,2024-02-09T08:50,08:50:00-05:00,9.0,42.450188,-71.120193,6.0,Coleoptera,NaN,True,false,8NBZS,Photinus corruscus,NaN,LC
3,HUMAN_OBSERVATION,2024-02-27T14:06:18,14:06:18-05:00,27.0,42.445595,-72.680771,1111.0,Coleoptera,NaN,True,false,XTZS,Conotrachelus carinifer,NaN,NaN
4,HUMAN_OBSERVATION,2024-03-01T17:30:54,17:30:54-05:00,1.0,44.662766,-74.997231,23.0,Coleoptera,NaN,True,false,PNVYQ,Harmonia axyridis,NaN,NaN


In [ ]:
df['eventDate'] = pd.to_datetime(df['eventDate'], errors='coerce').dt.date

In [ ]:
min_date = df['eventDate'].dropna().min()
max_date = df['eventDate'].dropna().max()
print(f"Minimum event date: {min_date}")
print(f"Maximum event date: {max_date}")

Minimum event date: 2024-01-01
Maximum event date: 2024-12-31


In [11]:
species_counts = df['species'].value_counts()
species_to_keep = species_counts[species_counts >= 20].index
df = df[df['species'].isin(species_to_keep)]
print(f"DataFrame filtered to {len(species_to_keep)} species with at least 20 observations.")
print(f"New DataFrame shape: {df.shape}")

DataFrame filtered to 2936 species with at least 20 observations.
New DataFrame shape: (774869, 15)


In [13]:
import requests
import time
import json

unique_species = df["species"].dropna().unique()
print(f"{len(unique_species)} unique species to look up")

common_name_map = {}
headers = {"User-Agent": "your-project-name (your-github-or-email)"}

for i, name in enumerate(unique_species):
    try:
        resp = requests.get(
            "https://api.inaturalist.org/v1/taxa",
            params={"q": name, "rank": "species", "per_page": 1},
            headers=headers
        )
        resp.raise_for_status()  # Raise an exception for HTTP errors (4xx or 5xx)
        results = resp.json().get("results", [])
        if results:
            common_name_map[name] = results[0].get("preferred_common_name") or name
        else:
            common_name_map[name] = name  # fall back to scientific name if no match found
    except requests.exceptions.RequestException as e:
        print(f"Request failed for species '{name}': {e}")
        if resp is not None:
            print(f"Status Code: {resp.status_code}")
            print(f"Response Text: {resp.text[:500]}...") # Print first 500 chars of response
        common_name_map[name] = name # Fallback to scientific name on error
    except json.JSONDecodeError as e:
        print(f"JSON Decode Error for species '{name}': {e}")
        if resp is not None:
            print(f"Status Code: {resp.status_code}")
            print(f"Response Text: {resp.text[:500]}...") # Print first 500 chars of response
        common_name_map[name] = name # Fallback to scientific name on error

    time.sleep(1)  # iNaturalist asks for ~1 request/second
    if (i + 1) % 100 == 0: # Print progress every 100 requests
        print(f"Processed {i+1} species...")

# save it so you never have to repeat this lookup
with open("common_names.json", "w") as f:
    json.dump(common_name_map, f)

df["common_name"] = df["species"].map(common_name_map)


2936 unique species to look up
Processed 100 species...
Processed 200 species...
Processed 300 species...
Processed 400 species...
Processed 500 species...
Processed 600 species...
Processed 700 species...
Processed 800 species...
Processed 900 species...
Processed 1000 species...
Processed 1100 species...
Processed 1200 species...
Processed 1300 species...
Processed 1400 species...
Processed 1500 species...
Processed 1600 species...
Processed 1700 species...
Processed 1800 species...
Processed 1900 species...
Processed 2000 species...
Processed 2100 species...
Processed 2200 species...
Processed 2300 species...
Processed 2400 species...
Processed 2500 species...
Processed 2600 species...
Processed 2700 species...
Processed 2800 species...
Processed 2900 species...


In [14]:
df.head()

,basisOfRecord,eventDate,eventTime,day,decimalLatitude,decimalLongitude,coordinateUncertaintyInMeters,order,elevation,hasCoordinate,hasGeospatialIssues,speciesKey,species,isInvasive,iucnRedListCategory,common_name
0,HUMAN_OBSERVATION,2024-01-08,09:41:10-05:00,8.0,46.232197,-67.793869,34.0,Hemiptera,NaN,True,false,6PGH7,Leptoglossus phyllopus,NaN,NaN,Eastern Leaf-footed Bug
1,HUMAN_OBSERVATION,NaT,17:54:00-05:00,24.0,43.022518,-77.060684,15.0,Lepidoptera,NaN,True,false,943K8,Hypena scabra,NaN,NaN,Green Cloverworm Moth
2,HUMAN_OBSERVATION,NaT,08:50:00-05:00,9.0,42.450188,-71.120193,6.0,Coleoptera,NaN,True,false,8NBZS,Photinus corruscus,NaN,LC,Winter Firefly
4,HUMAN_OBSERVATION,2024-03-01,17:30:54-05:00,1.0,44.662766,-74.997231,23.0,Coleoptera,NaN,True,false,PNVYQ,Harmonia axyridis,NaN,NaN,Asian Lady Beetle
5,HUMAN_OBSERVATION,2024-03-14,13:56:54-04:00,14.0,43.083748,-71.976891,6.0,Lepidoptera,NaN,True,false,486GN,Nymphalis antiopa,NaN,LC,Mourning Cloak


In [15]:
# Save the filtered DataFrame to Google Drive
df.to_csv('/content/drive/MyDrive/filtered_occurrence_data.csv', index=False)
print("DataFrame saved to /content/drive/MyDrive/filtered_occurrence_data.csv")

DataFrame saved to /content/drive/MyDrive/filtered_occurrence_data.csv


In [18]:
df["common_name"].value_counts()


,count
common_name,
Common Eastern Bumble Bee,15951
Asian Lady Beetle,11027
Western Honey Bee,10713
Spotted Lanternfly,10299
Monarch,9312
...,...
Antherophagus ochraceus,20
Black Locust Leafminer,20
Orange-shouldered Leafminer,20


In [ ]:
import geopandas as gpd
from shapely.geometry import Point


ecoregions = gpd.read_file("../data/us_eco_l3/us_eco_l3.shp").to_crs("EPSG:4326")

points = gpd.GeoDataFrame(
    df.copy(),
    geometry=[Point(xy) for xy in zip(df["decimalLongitude"], df["decimalLatitude"])],
    crs="EPSG:4326"
)

joined = gpd.sjoin(points, ecoregions[["geometry", "US_L3NAME"]], how="left", predicate="within")
df["ecoregion"] = joined["US_L3NAME"].values

missing = df["ecoregion"].isna()
if missing.any():
    nearest = gpd.sjoin_nearest(points[missing], ecoregions[["geometry", "US_L3NAME"]])
    df.loc[missing, "ecoregion"] = nearest["US_L3NAME"].values

print(df["ecoregion"].value_counts())

In [ ]:
import rasterio
from rasterio.warp import transform

with rasterio.open("../data/Annual_NLCD_LndCov_2024_CU_C1V1/Annual_NLCD_LndCov_2024_CU_C1V1.tif") as src:
    xs, ys = transform("EPSG:4326", src.crs, df["decimalLongitude"].tolist(), df["decimalLatitude"].tolist())
    land_cover_codes = [v[0] for v in src.sample(zip(xs, ys))]

df["land_cover_code"] = land_cover_codes


nlcd_labels = {
    11: "Open Water",
    12: "Perennial Ice/Snow",
    21: "Developed, Open Space",
    22: "Developed, Low Intensity",
    23: "Developed, Medium Intensity",
    24: "Developed, High Intensity",
    31: "Barren Land",
    41: "Deciduous Forest",
    42: "Evergreen Forest",
    43: "Mixed Forest",
    51: "Dwarf Scrub",        # Alaska only
    52: "Shrub/Scrub",
    71: "Grassland/Herbaceous",
    72: "Sedge/Herbaceous",   # Alaska only
    73: "Lichens",            # Alaska only
    74: "Moss",               # Alaska only
    81: "Pasture/Hay",
    82: "Cultivated Crops",
    90: "Woody Wetlands",
    95: "Emergent Herbaceous Wetlands",
}
df["land_cover_label"] = df["land_cover_code"].map(nlcd_labels)
print(df["land_cover_label"].value_counts())

In [ ]:
df.to_csv("../data/occurrence_ecoregions.csv", index=False)